# Mixing technique
> This technique is a mixing technique between
> - @***judith007*** https://www.kaggle.com/code/judith007/tuning-methods-included-better-score-lb1-3946
> - @***samu2505*** https://www.kaggle.com/code/samu2505/adding-location-features
> - @***tetsutani*** https://www.kaggle.com/code/tetsutani/catboost-only-tune-score-lb1-3845
> <br> ***Also thank to greate works from Jasonczh, Chris, Jasonczh, Giba, Nin7a1***

In [ ]:
import gc
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from haversine import haversine
from umap import UMAP
from typing import Tuple, List

from tqdm.notebook import tqdm

BASE = "../input/godaddy-microbusiness-density-forecasting/"


def smape(y_true, y_pred):
    smap = np.zeros(len(y_true))

    num = np.abs(y_true - y_pred)
    dem = (np.abs(y_true) + np.abs(y_pred)) / 2

    pos_ind = (y_true != 0) | (y_pred != 0)
    smap[pos_ind] = num[pos_ind] / dem[pos_ind]

    return 100 * np.mean(smap)


def vsmape(y_true, y_pred):
    smap = np.zeros(len(y_true))

    num = np.abs(y_true - y_pred)
    dem = (np.abs(y_true) + np.abs(y_pred)) / 2

    pos_ind = (y_true != 0) | (y_pred != 0)
    smap[pos_ind] = num[pos_ind] / dem[pos_ind]

    return 100 * smap

In [ ]:
census = pd.read_csv(BASE + "census_starter.csv")
print(census.columns)
census.head()

In [ ]:
train = pd.read_csv(BASE + "train.csv")
reaveal_test = pd.read_csv(BASE + "revealed_test.csv")
train = (
    pd.concat([train, reaveal_test])
    .sort_values(by=["cfips", "first_day_of_month"])
    .reset_index()
)
test = pd.read_csv(BASE + "test.csv")
drop_index = (test.first_day_of_month == "2022-11-01") | (
    test.first_day_of_month == "2022-12-01"
)
test = test.loc[~drop_index, :]
sub = pd.read_csv(BASE + "sample_submission.csv")
coords = pd.read_csv("/kaggle/input/usa-counties-coordinates/cfips_location.csv")
print(train.shape, test.shape, sub.shape)

train["istest"] = 0
test["istest"] = 1
raw = pd.concat((train, test)).sort_values(["cfips", "row_id"]).reset_index(drop=True)
raw = raw.merge(coords.drop("name", axis=1), on="cfips")

raw["state_i1"] = raw["state"].astype("category")
raw["county_i1"] = raw["county"].astype("category")
raw["first_day_of_month"] = pd.to_datetime(raw["first_day_of_month"])
raw["county"] = raw.groupby("cfips")["county"].ffill()
raw["state"] = raw.groupby("cfips")["state"].ffill()
raw["dcount"] = raw.groupby(["cfips"])["row_id"].cumcount()
raw["county_i"] = (raw["county"] + raw["state"]).factorize()[0]
raw["state_i"] = raw["state"].factorize()[0]
raw["scale"] = (raw["first_day_of_month"] - raw["first_day_of_month"].min()).dt.days
raw["scale"] = raw["scale"].factorize()[0]
# raw['population'] = np.round(np.mean(raw['active']*100/raw['microbusiness_density']))

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
raw.tail(20)

In [ ]:
test.first_day_of_month.unique()

In [ ]:
raw.loc[raw.dcount == (40), :].reset_index(drop=True)

# There are some anomalies, specially at timestep 18

In [ ]:
lag = 1
raw[f"mbd_lag_{lag}"] = raw.groupby("cfips")["microbusiness_density"].shift(lag).bfill()
raw["dif"] = (raw["microbusiness_density"] / raw[f"mbd_lag_{lag}"]).fillna(1).clip(
    0, None
) - 1
raw.loc[(raw[f"mbd_lag_{lag}"] == 0), "dif"] = 0
raw.loc[(raw[f"microbusiness_density"] > 0) & (raw[f"mbd_lag_{lag}"] == 0), "dif"] = 1
raw["dif"] = raw["dif"].abs()
raw.groupby("dcount")["dif"].sum().plot()

In [ ]:
outliers = []
cnt = 0
for o in tqdm(raw.cfips.unique()):
    indices = raw["cfips"] == o
    tmp = raw.loc[indices].copy().reset_index(drop=True)
    var = tmp.microbusiness_density.values.copy()
    # vmax = np.max(var[:38]) - np.min(var[:38])

    for i in range(40, 0, -1):
        thr = 0.20 * np.mean(var[:i])
        difa = abs(var[i] - var[i - 1])
        if difa >= thr:
            var[:i] *= var[i] / var[i - 1]
            outliers.append(o)
            cnt += 1
    var[0] = var[1] * 0.99
    raw.loc[indices, "microbusiness_density"] = var

outliers = np.unique(outliers)
len(outliers), cnt

In [ ]:
lag = 1
raw[f"mbd_lag_{lag}"] = raw.groupby("cfips")["microbusiness_density"].shift(lag).bfill()
raw["dif"] = (raw["microbusiness_density"] / raw[f"mbd_lag_{lag}"]).fillna(1).clip(
    0, None
) - 1
raw.loc[(raw[f"mbd_lag_{lag}"] == 0), "dif"] = 0
raw.loc[(raw[f"microbusiness_density"] > 0) & (raw[f"mbd_lag_{lag}"] == 0), "dif"] = 1
raw["dif"] = raw["dif"].abs()
raw.groupby("dcount")["dif"].sum().plot()

# SMAPE is a relative metric so target must be converted.

In [ ]:
raw["target"] = raw.groupby("cfips")["microbusiness_density"].shift(-1)
raw["target"] = raw["target"] / raw["microbusiness_density"] - 1


raw.loc[raw["cfips"] == 28055, "target"] = 0.0
raw.loc[raw["cfips"] == 48269, "target"] = 0.0

raw.iloc[:20, :20]

In [ ]:
raw["lastactive"] = raw.groupby("cfips")["active"].transform("last")

dt = raw.loc[raw.dcount == 40].groupby("cfips")["microbusiness_density"].agg("last")
raw["lasttarget"] = raw["cfips"].map(dt)

raw["lastactive"].clip(0, 8000).hist(bins=30)

# Feature Engineering

In [ ]:
def build_features(
    raw, target="microbusiness_density", target_act="active_tmp", lags=6
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Feature engineering
    
    Args:
        - raw (pd.DataFrame): Dataframe before preprocess
        - target (str): columns name to predict
        - target_act (str): columns name of active
        - lags (int): Lag count
    
    Returns:
        - pd.DataFrame: Preprocessed Dataframe
        - List[str]: List of column name to use
    """
    feats = []
    for lag in tqdm(range(1, lags)):
        raw[f"mbd_lag_{lag}"] = raw.groupby("cfips")[target].shift(lag)
        raw[f"act_lag_{lag}"] = raw.groupby("cfips")[target_act].diff(lag)
        feats.append(f"mbd_lag_{lag}")
        feats.append(f"act_lag_{lag}")

    lag = 1
    for window in [2, 4, 6, 8, 10, 12, 14]:
        raw[f"mbd_rollmea{window}_{lag}"] = raw.groupby("cfips")[
            f"mbd_lag_{lag}"
        ].transform(lambda s: s.rolling(window, min_periods=1).sum())
        # raw[f'mbd_rollmea{window}_{lag}'] = raw[f'mbd_lag_{lag}'] - raw[f'mbd_rollmea{window}_{lag}']
        feats.append(f"mbd_rollmea{window}_{lag}")

    census_columns = list(census.columns)
    census_columns.remove("cfips")

    raw = raw.merge(census, on="cfips", how="left")
    feats += census_columns

    return raw, feats

In [ ]:
# Build Features based in lag of target
raw, feats = build_features(raw, 'target', 'active', lags = 8)
features = ['state_i']
features += feats
features += ['lng','lat']
print(features)
raw.loc[raw.dcount==40, features].head(10)

In [ ]:
#state=raw.groupby(['state'])['target'].mean()
#raw['state_m'] = raw['state'].map(state)

#features += ['state_m']
features += ['scale']
features += ['lng','lat']

coordinates = raw[['lng', 'lat']].values

# Encoding tricks
emb_size = 20
precision = 1e6

latlon = np.expand_dims(coordinates, axis=-1)

m = np.exp(np.log(precision)/emb_size)
angle_freq = m ** np.arange(emb_size)
angle_freq = angle_freq.reshape(1,1, emb_size)
latlon = latlon * angle_freq
latlon[..., 0::2] = np.cos(latlon[..., 0::2])

!pip install -qq reverse_geocoder

import reverse_geocoder as rg

coordinates = list(zip(raw['lat'], raw['lng']))
results = rg.search(coordinates)
raw['place'] = [x['admin2'] for x in results]

places = list(np.unique(raw['county'].values))

def replace(x):
    if x in places:
        return x
    
    else:
        return 'Other'
    
raw['place'] = raw['place'].apply(lambda x: replace(x))

le = LabelEncoder()
raw['place'] = le.fit_transform(raw['place'])

pca = PCA(random_state=42).fit(coordinates)
raw['pca_lat'] = pca.transform(coordinates)[:, 0]
raw['pca_lon'] = pca.transform(coordinates)[:, 1]

umap = UMAP(n_components=2,
           n_neighbors=50,
           random_state=2023).fit(coordinates)

raw['umap_lat'] = umap.transform(coordinates)[:, 0]
raw['umap_lon'] = umap.transform(coordinates)[:, 1]

def rot(df):
    for angle in [15, 30, 45]:
        df[f'rot_{angle}_x'] = (np.cos(np.radians(angle)) * df['lat']) + \
                                (np.sin(np.radians(angle)) * df['lng'])
        
        df[f'rot_{angle}_y'] = (np.cos(np.radians(angle)) * df['lat']) - \
                                (np.sin(np.radians(angle)) * df['lng'])
        
    return df

raw = rot(raw)

features += ['place', 'rot_15_x', 'rot_15_y', 'rot_30_x', 'rot_30_y', 'rot_45_x', 'rot_45_y', 'pca_lat', 'pca_lon', 'umap_lat', 'umap_lon']
features = list(set(features))

# Set up a model

In [ ]:
# For type hinting
from sklearn.base import RegressorMixin


def get_model() -> RegressorMixin:
    """
    Previous model 
    
    Return:
        RegressorMixin: Regressor an instant class which has fit, predict
    """
    from sklearn.ensemble import VotingRegressor
    import lightgbm as lgb
    import xgboost as xgb
    import catboost as cat
    from sklearn.pipeline import Pipeline
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.impute import KNNImputer
    
    params = {
    'n_iter': 300,
    'verbosity': -1,
    'objective': 'l1',
    'random_state': 42,
    'colsample_bytree': 0.8841279649367693,
    'colsample_bynode': 0.10142964450634374,
    'max_depth': 8,
    'learning_rate': 0.003647749926797374,
    'lambda_l2': 0.5,
    'num_leaves': 61,
    "seed": 42,
    'min_data_in_leaf': 213}

    lgb_model = lgb.LGBMRegressor(**params)
    
    xgb_model = xgb.XGBRegressor(
        objective='reg:pseudohubererror',
        tree_method="hist",
        n_estimators=795,
        learning_rate=0.0075,
        max_leaves = 17,
        subsample=0.50,
        colsample_bytree=0.50,
        max_bin=4096,
        n_jobs=2,
        seed=42
#         eval_metric='mae',
#         early_stopping_rounds=70,
    )
# we should decrease the num_iterations of catboost
    cat_model = cat.CatBoostRegressor(
        iterations=800,
        loss_function="MAPE",
        verbose=0,
        grow_policy='SymmetricTree',
        learning_rate=0.035,
        max_depth=6,
        l2_leaf_reg=0.2,
#         max_leaves = 17,
        subsample=0.50,
        max_bin=4096,
#         delta=0.01
        random_seed=42
    )
    

#     knn_model = Pipeline([
#         ('imputer',  KNNImputer(n_neighbors=2)),
#         ('knn', KNeighborsRegressor(5))
#     ])
    
    return VotingRegressor([
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('cat', cat_model),
#         ('knn', knn_model)
    ],weights=[3,1,3])

def get_model() -> RegressorMixin:
    """
    Catboost model suggested by @tetsutani
    
    Return:
        RegressorMixin: Regressor an instant class which has fit, predict
    """
    from sklearn.ensemble import VotingRegressor
    import lightgbm as lgb
    import xgboost as xgb
    import catboost as cat
    from sklearn.pipeline import Pipeline
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.impute import KNNImputer
    
    params = {
    'n_iter': 300,
    'boosting_type': 'dart',
    'verbosity': -1,
    'objective': 'l1',
    'random_state': 42,
    'colsample_bytree': 0.8841279649367693,
    'colsample_bynode': 0.10142964450634374,
    'max_depth': 8,
    'learning_rate': 0.003647749926797374,
    'lambda_l2': 0.5,
    'num_leaves': 61,
    "seed": 42,
    'min_data_in_leaf': 213}

    lgb_model = lgb.LGBMRegressor(**params)
    
    xgb_model = xgb.XGBRegressor(
        objective='reg:pseudohubererror',
        tree_method="hist",
        n_estimators=795,
        learning_rate=0.0075,
        max_leaves = 17,
        subsample=0.50,
        colsample_bytree=0.50,
        max_bin=4096,
        n_jobs=2,
#         eval_metric='mae',
#         early_stopping_rounds=70,
    )
# we should decrease the num_iterations of catboost
    cat_model = cat.CatBoostRegressor(
        iterations=800,
        loss_function="MAPE",
        verbose=0,
        grow_policy='SymmetricTree',
        learning_rate=0.035,
        max_depth=5,
        l2_leaf_reg=0.2,
#         max_leaves = 17,
        subsample=0.50,
        max_bin=4096,
        random_seed=42  # add random seed to make it reproducable
#         delta=0.01
    )
    
# we should decrease the num_iterations of catboost
    cat_model2 = cat.CatBoostRegressor(
        iterations=800,
        loss_function="MAPE",
        verbose=0,
        grow_policy='SymmetricTree',
        learning_rate=0.035,
        max_depth=6,
        l2_leaf_reg=0.2,
#         max_leaves = 17,
        subsample=0.50,
        max_bin=4096,
        random_seed=43  # add random seed to make it reproducable
#         delta=0.01
    )
    
#     knn_model = Pipeline([
#         ('imputer',  KNNImputer(n_neighbors=2)),
#         ('knn', KNeighborsRegressor(5))
#     ])
    
    return VotingRegressor([
        #('xgb', xgb_model),
        #('lgb', lgb_model),
        ('cat', cat_model),
        ('cat2', cat_model2),
#         ('knn', knn_model)
    ],weights=[5,3])


# Start training
Note:
* blacklist seem not used any more, so I removed it

In [ ]:
ACT_THR = 140
ABS_THR = 0
raw["ypred_last"] = np.nan
raw["ypred"] = np.nan
raw["k"] = 1.0
VAL = []

for TS in range(39, 40):
    # Get model
    model = get_model()

    # Get train/test indices
    train_indices = (
        (raw.istest == 0)
        & (raw.dcount < TS)
        & (raw.dcount >= 1)
        & (raw.lastactive > ACT_THR)
        & (raw.lasttarget > ABS_THR)
    )
    valid_indices = (raw.istest == 0) & (raw.dcount == TS)

    # Fit
    model.fit(
        raw.loc[train_indices, features],
        raw.loc[train_indices, "target"].clip(-0.0043, 0.0045),
    )

    ypred = model.predict(raw.loc[valid_indices, features])
    
    # Predict
    raw.loc[valid_indices, "k"] = ypred + 1
    raw.loc[valid_indices, "k"] = (
        raw.loc[valid_indices, "k"] * raw.loc[valid_indices, "microbusiness_density"]
    )

    # Validate
    lastval = (
        raw.loc[raw.dcount == TS, ["cfips", "microbusiness_density"]]
        .set_index("cfips")
        .to_dict()["microbusiness_density"]
    )
    dt = raw.loc[raw.dcount == TS, ["cfips", "k"]].set_index("cfips").to_dict()["k"]

    df = raw.loc[
        raw.dcount == (TS + 1),
        ["cfips", "microbusiness_density", "state", "lastactive", "mbd_lag_1"],
    ].reset_index(drop=True)
    df["pred"] = df["cfips"].map(dt)
    df["lastval"] = df["cfips"].map(lastval)

    df.loc[df["lastactive"] <= ACT_THR, "pred"] = df.loc[
        df["lastactive"] <= ACT_THR, "lastval"
    ]
    df.loc[df["lastval"] <= ABS_THR, "pred"] = df.loc[
        df["lastval"] <= ABS_THR, "lastval"
    ]

    raw.loc[raw.dcount == (TS + 1), "ypred"] = df["pred"].values
    raw.loc[raw.dcount == (TS + 1), "ypred_last"] = df["lastval"].values
    pred = df["pred"].copy().fillna(2)
    last_val = df["lastval"].copy().fillna(2)
    print(f"TS: {TS}")
    print("Last Value SMAPE:", smape(df["microbusiness_density"], df["lastval"]))
    print("SMAPE:", smape(df["microbusiness_density"], df["pred"]))
    print()


ind = (raw.dcount >= 30) & (raw.dcount <= 40)
print("SMAPE:", smape(raw.loc[ind, "microbusiness_density"], raw.loc[ind, "ypred"]))
print(
    "Last Value SMAPE:",
    smape(raw.loc[ind, "microbusiness_density"], raw.loc[ind, "ypred_last"]),
)

# Predict test set
## Train all rows of training set

In [ ]:
TS = 40
print(TS)

model0 = get_model()

train_indices = (
    (raw.istest == 0)
    & (raw.dcount < TS)
    & (raw.dcount >= 1)
    & (raw.lastactive > ACT_THR)
    & (raw.lasttarget > ABS_THR)
)
valid_indices = raw.dcount == TS
model0.fit(
    raw.loc[train_indices, features],
    raw.loc[train_indices, "target"].clip(-0.0044, 0.0046),
)
ypred = model0.predict(raw.loc[valid_indices, features])
raw.loc[valid_indices, "k"] = ypred + 1.0
raw.loc[valid_indices, "k"] = (
    raw.loc[valid_indices, "k"] * raw.loc[valid_indices, "microbusiness_density"]
)

# Validate
lastval = (
    raw.loc[raw.dcount == TS, ["cfips", "microbusiness_density"]]
    .set_index("cfips")
    .to_dict()["microbusiness_density"]
)
dt = raw.loc[raw.dcount == TS, ["cfips", "k"]].set_index("cfips").to_dict()["k"]

In [ ]:
df = raw.loc[
    raw.dcount == (TS + 1),
    ["cfips", "microbusiness_density", "state", "lastactive", "mbd_lag_1"],
].reset_index(drop=True)
df

## Predict

In [ ]:
df["pred"] = df["cfips"].map(dt)
df["lastval"] = df["cfips"].map(lastval)
df.loc[df["lastactive"] <= ACT_THR, "pred"] = df.loc[
    df["lastactive"] <= ACT_THR, "lastval"
]
df.loc[df["lastval"] <= ABS_THR, "pred"] = df.loc[df["lastval"] <= ABS_THR, "lastval"]
raw.loc[raw.dcount == (TS + 1), "ypred"] = df["pred"].values
raw.loc[raw.dcount == (TS + 1), "ypred_last"] = df["lastval"].values

In [ ]:
raw[["cfips", "microbusiness_density", "dcount", "ypred", "ypred_last", "k"]].tail(20)

In [ ]:
raw.loc[raw["cfips"] == 28055, "microbusiness_density"] = 0
raw.loc[raw["cfips"] == 48269, "microbusiness_density"] = 1.762115

dt = raw.loc[raw.dcount == 41, ["cfips", "ypred"]].set_index("cfips").to_dict()["ypred"]
test = raw.loc[raw.istest == 1, ["row_id", "cfips", "microbusiness_density"]].copy()
test

In [ ]:
test["microbusiness_density"] = test["cfips"].map(dt)

test = test[["row_id", "microbusiness_density"]]
test

## Finalize submission

In [ ]:
from IPython.display import display


sample_sub = pd.read_csv(BASE + "revealed_test.csv")
sub_index = (sample_sub.first_day_of_month == "2022-11-01") | (
    sample_sub.first_day_of_month == "2022-12-01"
)
test1 = pd.concat(
    [
        sample_sub.loc[sub_index, :]
        .drop(
            [
                i
                for i in sample_sub.columns
                if i != "row_id" and i != "microbusiness_density"
            ],
            axis=1,
        )
        .fillna(2),
        test,
    ]
)

test1 = test1.fillna(1.7569)

COLS = ["GEO_ID", "NAME", "S0101_C01_026E"]
df2020 = pd.read_csv(
    "/kaggle/input/census-data-for-godaddy/ACSST5Y2020.S0101-Data.csv", usecols=COLS
)
df2020 = df2020.iloc[1:]
df2020["S0101_C01_026E"] = df2020["S0101_C01_026E"].astype("int")


df2021 = pd.read_csv(
    "/kaggle/input/census-data-for-godaddy/ACSST5Y2021.S0101-Data.csv", usecols=COLS
)
df2021 = df2021.iloc[1:]
df2021["S0101_C01_026E"] = df2021["S0101_C01_026E"].astype("int")
print(df2021.shape)
df2021.head()

test1["cfips"] = test1.row_id.apply(lambda x: int(x.split("_")[0]))

test1.head()

df2020["cfips"] = df2020.GEO_ID.apply(lambda x: int(x.split("US")[-1]))
adult2020 = df2020.set_index("cfips").S0101_C01_026E.to_dict()

df2021["cfips"] = df2021.GEO_ID.apply(lambda x: int(x.split("US")[-1]))
adult2021 = df2021.set_index("cfips").S0101_C01_026E.to_dict()

test1["adult2020"] = test1.cfips.map(adult2020)
test1["adult2021"] = test1.cfips.map(adult2021)
test1.head()

test1.microbusiness_density = (
    test1.microbusiness_density * test1.adult2020 / test1.adult2021
)
test1 = test1.drop(["adult2020", "adult2021", "cfips"], axis=1)
test1.to_csv("submission_test1.csv", index=False)
test1.head()